# ExoJump cloud model benchmark

Run the leakage-aware classical, CNN, BiLSTM, TCN, MiniROCKET, and modality-ablation experiments in Google Colab. Select a GPU runtime before starting. Only upload the prepared anonymised `jump_event_windows.npz`; do not upload raw participant files or videos.

## 1. Load the private bundle and install

The default, most reliable route is to upload `exojump-code.zip` and `jump_event_windows.npz` through Colab's Files pane, then keep `USE_DRIVE=False`. Set `USE_DRIVE=True` only when Drive mounting works. Neither route requires granting Colab access to the private GitHub account.

In [ ]:
from pathlib import Path
import shutil, subprocess, sys

USE_DRIVE = False
DRIVE_ROOT = Path('/content/drive/MyDrive/exojump_private')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
BUNDLE = DRIVE_ROOT / 'exojump-code.zip' if USE_DRIVE else Path('/content/exojump-code.zip')
PROJECT = Path('/content/exoskeleton-jump-recognition')
assert BUNDLE.exists(), f'Upload the code bundle to {BUNDLE}'
if PROJECT.exists():
    shutil.rmtree(PROJECT)
shutil.unpack_archive(str(BUNDLE), '/content')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e',
    f'{PROJECT}[analysis,deep-learning,time-series]'
], check=True)
print('Project ready:', PROJECT)

## 2. Select the prepared dataset

With `USE_DRIVE=False`, upload `jump_event_windows.npz` to the Colab session through the Files pane. With `USE_DRIVE=True`, keep it in `MyDrive/exojump_private/`. Drive-mode outputs survive a disconnect; session-mode outputs must be downloaded before the runtime closes.

In [ ]:
DATASET = DRIVE_ROOT / 'jump_event_windows.npz' if USE_DRIVE else Path('/content/jump_event_windows.npz')
OUTPUT_ROOT = DRIVE_ROOT / 'cloud_results' if USE_DRIVE else Path('/content/cloud_results')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
assert DATASET.exists(), f'Upload the prepared NPZ to {DATASET}'
print('Dataset:', DATASET)
print('Outputs:', OUTPUT_ROOT)

In [ ]:
import numpy as np, torch
arrays = np.load(DATASET, allow_pickle=False)
print({key: arrays[key].shape for key in arrays.files})
print('Participants:', sorted(np.unique(arrays['subject'].astype(str))))
print('CUDA available:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 3. Choose quick or full mode

Keep `QUICK_MODE=True` for the first run. After it succeeds, set it to `False` for the reportable three-seed experiment.

In [ ]:
QUICK_MODE = True
SEEDS = '42' if QUICK_MODE else '42,7,123'
EPOCHS = '3' if QUICK_MODE else '30'
PATIENCE = '2' if QUICK_MODE else '5'
CLASSICAL_MODELS = 'logistic,svm' if QUICK_MODE else 'logistic,svm,random_forest,minirocket'
print({'quick': QUICK_MODE, 'seeds': SEEDS, 'epochs': EPOCHS, 'device': DEVICE})

## 4. Classical and MiniROCKET baselines

In [ ]:
classical_output = OUTPUT_ROOT / 'classical_benchmark.json'
subprocess.run([
    sys.executable, str(PROJECT / 'scripts/benchmark_classical_models.py'),
    '--dataset', str(DATASET), '--output', str(classical_output),
    '--models', CLASSICAL_MODELS, '--modalities', 'fusion',
    '--bootstrap-resamples', '200' if QUICK_MODE else '2000'
], cwd=PROJECT, check=True)

## 5. CNN, BiLSTM, and TCN comparison

In [ ]:
architecture_output = OUTPUT_ROOT / 'deep_architectures'
subprocess.run([
    sys.executable, str(PROJECT / 'scripts/benchmark_deep_models.py'),
    '--dataset', str(DATASET), '--output', str(architecture_output),
    '--architectures', 'cnn,bilstm,tcn', '--modalities', 'fusion',
    '--seeds', SEEDS, '--epochs', EPOCHS, '--patience', PATIENCE,
    '--device', DEVICE, '--bootstrap-resamples', '200' if QUICK_MODE else '2000'
], cwd=PROJECT, check=True)

## 6. CNN modality ablation

The fusion CNN was trained in the previous cell, so this cell adds only IMU-only and sEMG-only controls.

In [ ]:
ablation_output = OUTPUT_ROOT / 'cnn_modality_ablation'
subprocess.run([
    sys.executable, str(PROJECT / 'scripts/benchmark_deep_models.py'),
    '--dataset', str(DATASET), '--output', str(ablation_output),
    '--architectures', 'cnn', '--modalities', 'imu,semg',
    '--seeds', SEEDS, '--epochs', EPOCHS, '--patience', PATIENCE,
    '--device', DEVICE, '--bootstrap-resamples', '200' if QUICK_MODE else '2000'
], cwd=PROJECT, check=True)

## 7. Produce the report table

In [ ]:
summary_path = OUTPUT_ROOT / 'MODEL_COMPARISON.md'
subprocess.run([
    sys.executable, str(PROJECT / 'scripts/summarize_benchmarks.py'),
    '--classical', str(classical_output),
    '--deep', str(architecture_output / 'deep_benchmark.json'),
    '--output', str(summary_path)
], cwd=PROJECT, check=True)
print(summary_path.read_text())
archive_base = OUTPUT_ROOT.parent / 'exojump_benchmark_results'
archive_path = shutil.make_archive(str(archive_base), 'zip', OUTPUT_ROOT)
print('Results archive:', archive_path)